# D364 Exercise: Olist customer and product segmentation on Amazon Redshift

## Business use case

Olist wants a **retention and assortment strategy**. The CRM team needs actionable customer segments, while the merchandising team needs product-category segments based on demand, revenue, delivery experience, and reviews. Build the solution with native tables on an **Amazon Redshift provisioned cluster**.

You already know S3, IAM roles, and `COPY`. This notebook therefore gives only the commands needed for this dataset. Run Bash commands in a terminal and SQL in Redshift Query Editor v2. Replace every value in angle brackets. Do not paste placeholders literally.

### Learning outcomes

- classify fact-like and dimensional tables;
- choose `DISTSTYLE ALL` or `DISTSTYLE KEY` and an appropriate `DISTKEY`;
- choose sort keys from the actual join and filter workload;
- build customer and product segments without double-counting;
- validate physical design with Redshift system views and `EXPLAIN`.


## 1. Source files and analytical grain

Use these files from `C:\data\olist`:

| File | Intended grain | Important columns |
|---|---|---|
| `olist_customers_dataset.csv` | one row per order-level customer ID | `customer_id`, `customer_unique_id`, state |
| `olist_orders_dataset.csv` | one row per order | status and purchase/delivery timestamps |
| `olist_order_items_dataset.csv` | one row per order line | product, seller, price, freight |
| `olist_products_dataset.csv` | one row per product | category and physical attributes |
| `olist_order_reviews_dataset.csv` | one row per submitted review | order, score, review timestamps |
| `product_category_name_translation.csv` | one row per category translation | Portuguese and English category names |

> Important: `customer_id` identifies a customer record associated with an order. Use `customer_unique_id` to recognize the same shopper across orders. Revenue is calculated at **order-item grain** as `price + freight_value`; aggregate it before joining to data that may contain multiple rows per order.


## 2. Upload only the required Olist files to S3

The following PowerShell commands copy the six required files; they do not synchronize or delete other objects. Use a bucket in the same Region as the Redshift cluster.
Copy below files to S3
~~~ 
  'olist_customers_dataset.csv',
  'olist_orders_dataset.csv',
  'olist_order_items_dataset.csv',
  'olist_products_dataset.csv',
  'olist_order_reviews_dataset.csv',
  'product_category_name_translation.csv'
 
~~~

Record the bucket, Region, and six object names in your submission. Confirm that your existing Redshift S3 role can read this prefix and is attached to the provisioned cluster.


## 3. Physical-design decision (complete before writing DDL)

Classify each table as fact-like or dimensional. For every table, choose one of the following designs:

- `DISTSTYLE ALL` for a sufficiently small, frequently joined dimension that is worth copying to every compute node; or
- `DISTSTYLE KEY` plus one `DISTKEY` for a larger table whose rows should be colocated for the dominant joins.

Then choose a `COMPOUND SORTKEY` (one or more columns). Base the leading column on common range filters or join/aggregation access patterns—not merely on primary-key convention. Do not use `AUTO` for this exercise. A column can be only one table's distribution key, so explain which expensive join you prioritized when multiple relationships compete.

Complete this table in a Markdown cell in your submitted notebook:

| Table | Fact/dimension | Estimated rows | ALL or KEY | DISTKEY if KEY | Sort key(s) | Workload justification |
|---|---:|---:|---|---|---|---|
| customers | | | | | | |
| orders | | | | | | |
| order_items | | | | | | |
| products | | | | | | |
| order_reviews | | | | | | |
| category_translation | | | | | | |


## 4. Create the schema and native tables

Create all six tables in schema `olist`. Preserve the source column order so that `COPY` can load without a column list. Replace each `<physical-design-clause>` with either your `DISTSTYLE ALL` design or your `DISTSTYLE KEY DISTKEY (...)` design, followed by your sort key.

~~~sql
CREATE SCHEMA IF NOT EXISTS olist;

CREATE TABLE olist.customers (
  customer_id              VARCHAR(32) NOT NULL,
  customer_unique_id       VARCHAR(32) NOT NULL,
  customer_zip_code_prefix INTEGER,
  customer_city            VARCHAR(100),
  customer_state           CHAR(2)
) <physical-design-clause>;

CREATE TABLE olist.orders (
  order_id                         VARCHAR(32) NOT NULL,
  customer_id                      VARCHAR(32) NOT NULL,
  order_status                     VARCHAR(20),
  order_purchase_timestamp         TIMESTAMP,
  order_approved_at                TIMESTAMP,
  order_delivered_carrier_date     TIMESTAMP,
  order_delivered_customer_date    TIMESTAMP,
  order_estimated_delivery_date    TIMESTAMP
) <physical-design-clause>;

CREATE TABLE olist.order_items (
  order_id            VARCHAR(32) NOT NULL,
  order_item_id        INTEGER NOT NULL,
  product_id           VARCHAR(32) NOT NULL,
  seller_id            VARCHAR(32),
  shipping_limit_date  TIMESTAMP,
  price                DECIMAL(12,2),
  freight_value        DECIMAL(12,2)
) <physical-design-clause>;

CREATE TABLE olist.products (
  product_id                    VARCHAR(32) NOT NULL,
  product_category_name         VARCHAR(100),
  product_name_lenght           INTEGER,
  product_description_lenght    INTEGER,
  product_photos_qty            INTEGER,
  product_weight_g              INTEGER,
  product_length_cm             INTEGER,
  product_height_cm             INTEGER,
  product_width_cm              INTEGER
) <physical-design-clause>;

CREATE TABLE olist.order_reviews (
  review_id               VARCHAR(32) NOT NULL,
  order_id                VARCHAR(32) NOT NULL,
  review_score            INTEGER,
  review_comment_title    VARCHAR(500),
  review_comment_message  VARCHAR(5000),
  review_creation_date    TIMESTAMP,
  review_answer_timestamp TIMESTAMP
) <physical-design-clause>;

CREATE TABLE olist.category_translation (
  product_category_name         VARCHAR(100) NOT NULL,
  product_category_name_english VARCHAR(100)
) <physical-design-clause>;
~~~

The source deliberately retains Olist's misspelled `lenght` column names. If rerunning, drop only these exercise tables or use a fresh schema; do not blindly use `CASCADE` in a shared cluster.


## 5. Validate and load from S3 with COPY

First run each command with `NOLOAD`. When all six validations pass, remove `NOLOAD` and run the commands again. If your cluster has the S3 role configured as its default role, `IAM_ROLE default` is sufficient.

~~~sql
COPY olist.customers
FROM 's3://<bucket-name>/redshift/olist/olist_customers_dataset.csv'
IAM_ROLE default REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL TRUNCATECOLUMNS
COMPUPDATE ON STATUPDATE ON NOLOAD;

COPY olist.orders
FROM 's3://<bucket-name>/redshift/olist/olist_orders_dataset.csv'
IAM_ROLE default REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL TIMEFORMAT 'auto'
COMPUPDATE ON STATUPDATE ON NOLOAD;

COPY olist.order_items
FROM 's3://<bucket-name>/redshift/olist/olist_order_items_dataset.csv'
IAM_ROLE default REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL TIMEFORMAT 'auto'
COMPUPDATE ON STATUPDATE ON NOLOAD;

COPY olist.products
FROM 's3://<bucket-name>/redshift/olist/olist_products_dataset.csv'
IAM_ROLE default REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL
COMPUPDATE ON STATUPDATE ON NOLOAD;

COPY olist.order_reviews
FROM 's3://<bucket-name>/redshift/olist/olist_order_reviews_dataset.csv'
IAM_ROLE default REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL TIMEFORMAT 'auto' TRUNCATECOLUMNS
COMPUPDATE ON STATUPDATE ON NOLOAD;

COPY olist.category_translation
FROM 's3://<bucket-name>/redshift/olist/product_category_name_translation.csv'
IAM_ROLE default REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL
COMPUPDATE ON STATUPDATE ON NOLOAD;
~~~

If the role is not the cluster default, replace `default` with its full ARN. Do not use `ACCEPTINVCHARS` until you have inspected the failing source value and documented why accepting replacement characters is appropriate.


## 6. Ingestion quality gate

Write SQL that proves the load is usable before analysis. Your output must include:

1. one row count for each of the six tables;
2. duplicate checks at the declared grain of each table;
3. orphan counts for customers-to-orders, orders-to-items, products-to-items, and orders-to-reviews;
4. minimum and maximum business timestamps;
5. invalid values: negative price/freight, review scores outside 1–5, and unknown order statuses.

Use `SYS_LOAD_HISTORY` and `SYS_LOAD_ERROR_DETAIL` to diagnose failed loads. Capture the validation SQL and its result in your submission.

~~~sql
-- Starting point: add all six tables.
SELECT 'customers' AS table_name, COUNT(*) AS row_count
FROM olist.customers;

SELECT query_id, table_name, status, loaded_rows, error_count, start_time
FROM sys_load_history
WHERE start_time >= DATEADD(hour, -4, GETDATE())
ORDER BY start_time DESC;
~~~


## 7. Build a reusable order-level foundation

Create a view named `olist.v_order_metrics` with **exactly one row per order**. It must contain at least:

- order and customer identifiers, status, purchase date, and customer state;
- item count and distinct-product count;
- item revenue, freight revenue, and total order value;
- delivered days and delivery-delay days;
- one review score per order, using a documented rule when duplicate reviews exist.

Requirements:

- aggregate order items and reviews in separate CTEs before joining;
- do not allow a many-to-many join to inflate revenue;
- preserve orders without items or reviews where analytically useful;
- define whether cancelled/unavailable orders belong in revenue metrics.

Prove the view's grain with `COUNT(*)`, `COUNT(DISTINCT order_id)`, and a duplicate query.


## 8. Customer segmentation exercises

Use `customer_unique_id` as the customer grain. Use the dataset's maximum purchase timestamp as the analysis date so results remain reproducible.

### C1 — Customer 360
Create `olist.v_customer_360` with one row per unique customer: first purchase, last purchase, recency days, completed-order frequency, total item revenue, freight, total value, average order value, average review score, average delivery days, and late-delivery rate. Validate its grain.

### C2 — RFM scores
Use `NTILE(5)` to score recency, frequency, and monetary value. Ensure a high score always means desirable behavior; explicitly handle ties and NULLs. Return `r_score`, `f_score`, `m_score`, and an RFM code.

### C3 — Actionable CRM segments
Use `CASE` to map customers into at least: Champions, Loyal, Potential Loyalists, New Customers, At Risk, Hibernating, and Other. State the score rules. Return customer count, revenue, average order value, and revenue share per segment.

### C4 — Geographic comparison
For each customer segment and state, calculate customers, revenue, average delivery days, late-delivery rate, and average review. Rank states within each segment by revenue. Suppress or flag groups with fewer than 30 completed orders.

### C5 — Repeat-purchase cohort
Assign customers to a first-purchase month. Calculate the percentage placing a second completed order within 30, 60, and 90 days. Clearly define numerator and denominator and avoid counting multiple later orders more than once.

### C6 — Retention target list
Produce a prioritized list of at-risk customers who historically generated above-median customer revenue. Include their last purchase date, recency, order count, value, favorite English product category, and service indicators. Explain the outreach priority.


## 9. Product and category segmentation exercises

Use English category names where available and a clear fallback such as `Untranslated/Unknown`. State whether freight is included in product revenue.

### P1 — Product 360
Create `olist.v_product_360` with one row per product: English category, units sold, distinct orders, revenue, first/last sale, active selling months, average review score, late-delivery rate, and dimensions/weight. Prevent order reviews from multiplying item revenue.

### P2 — ABC revenue segmentation
Rank products by revenue and calculate cumulative revenue share with window functions. Label products A (first 80%), B (next 15%), or C (remaining 5%). Document how the boundary-crossing product is classified. Report product and revenue share by class.

### P3 — Demand versus experience matrix
Within each category, score product demand and customer experience into quartiles. Create four business segments such as Stars, Hidden Gems, Fix Experience, and Low Priority. Use a minimum-order threshold before interpreting review averages.

### P4 — Category growth
Aggregate monthly category revenue, use `LAG` to calculate month-over-month growth, and compare the latest complete month with the previous month. Return the top five growing and bottom five declining categories, excluding categories below a justified revenue threshold.

### P5 — Freight burden
For each category, calculate freight as a percentage of item price and segment categories into Low, Medium, and High freight burden using dataset-driven thresholds. Compare by customer state and identify combinations that may harm margin or conversion.

### P6 — Assortment recommendation
Recommend categories to Grow, Maintain, Fix, or Rationalize using revenue, growth, review score, late-delivery rate, and freight burden. Provide a ranked output and a one-sentence business rationale for the top ten recommendations.


## 10. Redshift physical-design verification

After loading and running the analytical queries, inspect whether your design behaved as intended.

~~~sql
ANALYZE olist.customers;
ANALYZE olist.orders;
ANALYZE olist.order_items;
ANALYZE olist.products;
ANALYZE olist.order_reviews;
ANALYZE olist.category_translation;

SELECT "table", diststyle, sortkey1, sortkey_num, size, tbl_rows,
       skew_rows, unsorted, stats_off
FROM svv_table_info
WHERE "schema" = 'olist'
ORDER BY "table";
~~~

Run `EXPLAIN` for one customer query and one product query. Identify redistribution/broadcast indicators such as `DS_DIST_*`, then answer:

1. Which joins are colocated, broadcast, or redistributed?
2. Is any KEY-distributed table skewed? Give evidence.
3. Are the chosen leading sort columns aligned with selective filters?
4. Would changing one table from ALL to KEY, or KEY to ALL, improve this workload? Explain storage and load-time tradeoffs.
5. If you recommend a redesign, create a comparison table with the alternative physical design, load it, and compare both plans.


## 11. Submission checklist

Submit one notebook or SQL file containing:

- the completed physical-design decision table and justification;
- executable DDL with no unresolved placeholders;
- COPY commands and ingestion-quality results;
- `v_order_metrics`, `v_customer_360`, and `v_product_360`;
- solutions for C1–C6 and P1–P6;
- query results or screenshots for the principal segment outputs;
- `SVV_TABLE_INFO` evidence and two annotated `EXPLAIN` plans;
- a short recommendation to the CRM team and the merchandising team.

Assessment emphasizes correct grain, prevention of double-counting, defensible distribution/sort choices, reproducible SQL, and business usefulness—not just query completion.


## Optional cleanup

Run only in your own exercise schema after assessment.

~~~sql
-- DROP SCHEMA olist CASCADE;
~~~
